# **Period Regime Finder**

## **1. Project Setup**

In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\candle-fetcher


## **2. Imports**

In [12]:
from pathlib import Path

from src.mt5_fetcher import MT5FetchConfig, fetch_raw_candles, print_fetch_result

from src.regime_period_filter import (
    find_period_regimes_from_csv,
    format_period_regime_table,
    print_period_regime_results,
)

## **3. Settings**

In [ ]:
# ---------------------------------------------------------------------------
# Period Regime Finder Settings
# ---------------------------------------------------------------------------

# Choose one:
# "year", "month", "week", "day"
PERIOD = "month"

# Choose one:
# "calm", "volatile"
REGIME = "calm"

TOP_N = 10


# ---------------------------------------------------------------------------
# Data Source Settings
# ---------------------------------------------------------------------------

# False = use an existing CSV
# True = fetch fresh candles from MT5/FTMO first
FETCH_FROM_MT5 = False

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "all_sessions" / "us100_cash_M1_all_sessions_2024_01_01_to_2024_12_31.csv"


# ---------------------------------------------------------------------------
# MT5 Fetch Settings
# ---------------------------------------------------------------------------

SYMBOL = "US100.cash"

# You can write "1m", "5m", "15m", "1h", etc.
# It will be converted internally to MT5 format.
TIMEFRAME = "1m"

FETCH_START_DATE = "2024-01-01"
FETCH_END_DATE = "2024-12-31"

RANGE_NAME = "period_regime_range"
RAW_OUTPUT_NAME = None

CHUNK_DAYS = 31

# For all-session regime analysis, UTC is simplest.
DATE_TIMEZONE_NAME = "UTC"


# ---------------------------------------------------------------------------
# Output Settings
# ---------------------------------------------------------------------------

SAVE_RESULTS = True

OUTPUT_FOLDER = PROJECT_ROOT / "data" / "comparisons" / "tables"

OUTPUT_NAME = None

## **4. Fetch Raw Candles From MT5**


In [ ]:
input_csv_path = CSV_PATH

if FETCH_FROM_MT5:
    fetch_config = MT5FetchConfig(
        symbol=SYMBOL,
        timeframe=TIMEFRAME,
        range_name=RANGE_NAME,
        output_name=RAW_OUTPUT_NAME,
        start_date=FETCH_START_DATE,
        end_date=FETCH_END_DATE,
        chunk_days=CHUNK_DAYS,
        date_timezone_name=DATE_TIMEZONE_NAME,
    )

    fetch_result = fetch_raw_candles(fetch_config)
    print_fetch_result(fetch_result)

    input_csv_path = Path(fetch_result["output_path"])

else:
    print("Skipping MT5 fetch. Using existing CSV.")
    print(f"CSV path: {input_csv_path}")

Skipping MT5 fetch. Using existing CSV.
CSV path: c:\candle-fetcher\data\processed\all_sessions\us100_cash_M1_all_sessions_2024_01_01_to_2024_12_31.csv


## **5. Run Period Regime Filter**


In [15]:
result = find_period_regimes_from_csv(
    csv_path=input_csv_path,
    period=PERIOD,
    regime=REGIME,
    top_n=TOP_N,
)

print_period_regime_results(result)

Period Regime Results
Lower volatility score = calmer period.
Higher volatility score = more volatile period.

 Rank Period Type  Period  Rows              First Candle               Last Candle Regime Label  Volatility Score  Average Candle Range  Median Candle Range  Largest Candle Range  Return Volatility  Average Tick Volume  Average Spread
    1       month 2024-06 27069 2024-06-03 01:05:00+00:00 2024-06-28 23:49:00+00:00         calm              19.2                  4.86                  3.4                 100.7           0.000223                 98.0          157.31
    2       month 2024-05 31165 2024-05-01 01:05:00+00:00 2024-05-31 23:49:00+00:00         calm              26.7                  4.20                  2.9                 181.5           0.000214                109.0          150.05
    3       month 2024-01 29741 2024-01-02 01:05:00+00:00 2024-01-31 23:49:00+00:00         calm              28.3                  4.68                  3.4                 111.2  

## **6. View Results**

In [9]:
friendly_df = format_period_regime_table(result)

friendly_df

,Rank,Period Type,Period,Rows,First Candle,Last Candle,Regime Label,Volatility Score,Average Candle Range,Median Candle Range,Largest Candle Range,Return Volatility,Average Tick Volume,Average Spread
0,1,month,2024-06,27069,2024-06-03 01:05:00+00:00,2024-06-28 23:49:00+00:00,calm,19.2,4.86,3.4,100.7,0.000223,98.0,157.31
1,2,month,2024-05,31165,2024-05-01 01:05:00+00:00,2024-05-31 23:49:00+00:00,calm,26.7,4.20,2.9,181.5,0.000214,109.0,150.05
2,3,month,2024-01,29741,2024-01-02 01:05:00+00:00,2024-01-31 23:49:00+00:00,calm,28.3,4.68,3.4,111.2,0.000233,124.0,147.72
3,4,month,2024-02,28430,2024-02-01 01:05:00+00:00,2024-02-29 23:49:00+00:00,calm,30.8,4.70,3.4,161.8,0.000236,117.0,144.90
4,5,month,2024-03,27240,2024-03-01 01:05:00+00:00,2024-03-28 23:49:00+00:00,normal,40.0,5.18,3.7,107.6,0.000250,120.0,147.20
5,6,month,2024-12,28449,2024-12-02 01:05:00+00:00,2024-12-31 23:49:00+00:00,normal,50.0,6.25,4.3,88.0,0.000271,160.0,159.20
6,7,month,2024-10,31395,2024-10-01 01:05:00+00:00,2024-10-31 23:49:00+00:00,normal,53.3,6.41,4.8,106.3,0.000259,164.0,168.19
7,8,month,2024-11,28184,2024-11-01 01:05:00+00:00,2024-11-29 20:14:00+00:00,normal,65.0,6.45,5.0,175.4,0.000262,186.0,166.44
8,9,month,2024-07,30778,2024-07-01 01:05:00+00:00,2024-07-31 23:49:00+00:00,volatile,70.0,6.55,4.4,115.3,0.000308,158.0,167.91
9,10,month,2024-04,30019,2024-04-01 01:05:00+00:00,2024-04-30 23:49:00+00:00,volatile,78.3,6.11,4.5,241.9,0.000308,132.0,164.28


## **7. Check Size**

In [10]:
result.shape

(10, 31)

## **8. Save Results**

In [11]:
if SAVE_RESULTS:
    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

    if OUTPUT_NAME is None:
        output_name = f"period_regime_{PERIOD}_{REGIME}_top_{TOP_N}.csv"
    else:
        output_name = OUTPUT_NAME

        if not output_name.lower().endswith(".csv"):
            output_name += ".csv"

    output_path = OUTPUT_FOLDER / output_name

    friendly_df.to_csv(output_path, index=False)

    print(f"Saved results to: {output_path}")
else:
    print("SAVE_RESULTS is False. Results were not saved.")

Saved results to: c:\candle-fetcher\data\comparisons\tables\period_regime_month_calm_top_10.csv
